In [2]:
import sys
print(sys.version)
print(sys.executable)

3.11.16 (main, Aug 27 2026, 14:37:43) [Clang 20.1.8 ]
/opt/anaconda3/envs/marketing-incrementality-py311/bin/python


In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/hillstrom.csv")

df.head()

df.shape

(64000, 12)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  str    
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  str    
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  str    
 8   segment          64000 non-null  str    
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), str(4)
memory usage: 5.9 MB


In [3]:
for col in ["history_segment", "zip_code", "channel", "segment"]:
    print(f"\n{col}:")
    print(df[col].unique())


history_segment:
<StringArray>
[  '2) $100 - $200',   '3) $200 - $350',   '5) $500 - $750',
     '1) $0 - $100', '6) $750 - $1,000',   '4) $350 - $500',
      '7) $1,000 +']
Length: 7, dtype: str

zip_code:
<StringArray>
['Surburban', 'Rural', 'Urban']
Length: 3, dtype: str

channel:
<StringArray>
['Phone', 'Web', 'Multichannel']
Length: 3, dtype: str

segment:
<StringArray>
['Womens E-Mail', 'No E-Mail', 'Mens E-Mail']
Length: 3, dtype: str


In [4]:
df.isna().sum()

recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(6562)

In [6]:
df["segment"].value_counts()


segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

In [7]:
df.groupby("segment")[["visit", "conversion", "spend"]].mean()

,visit,conversion,spend
segment,,,
Mens E-Mail,0.182757,0.012531,1.422617
No E-Mail,0.106167,0.005726,0.652789
Womens E-Mail,0.151400,0.008837,1.077202


In [8]:
# Create the control group:
# these are customers who received no marketing email.
control = df[df["segment"] == "No E-Mail"]

# Compare each treatment group against the control group.
for treatment in ["Mens E-Mail", "Womens E-Mail"]:

    # Select only the customers who received the current treatment.
    treated = df[df["segment"] == treatment]

    # Calculate absolute lift for each business outcome.
    # Lift = treatment average - control average
    # Positive lift means the treatment performed better than no email.

    visit_lift = treated["visit"].mean() - control["visit"].mean()
    conversion_lift = treated["conversion"].mean() - control["conversion"].mean()
    spend_lift = treated["spend"].mean() - control["spend"].mean()

    # Print the results clearly for each treatment.
    print(f"\n{treatment}")
    print("Visit lift:", visit_lift)
    print("Conversion lift:", conversion_lift)
    print("Spend lift:", spend_lift)


Mens E-Mail
Visit lift: 0.07658956365153125
Conversion lift: 0.006805006519615695
Spend lift: 0.7698271558945368

Womens E-Mail
Visit lift: 0.045233106587052985
Conversion lift: 0.003111057511085258
Spend lift: 0.4244122159365967


In [9]:
from scipy import stats

# Separate the three experimental groups.
control = df[df["segment"] == "No E-Mail"]
mens = df[df["segment"] == "Mens E-Mail"]
womens = df[df["segment"] == "Womens E-Mail"]

# Welch's t-test compares the average spend between two groups.
# We use Welch's version because it does not assume both groups
# have exactly the same variance.

mens_spend_test = stats.ttest_ind(
    mens["spend"],
    control["spend"],
    equal_var=False
)

womens_spend_test = stats.ttest_ind(
    womens["spend"],
    control["spend"],
    equal_var=False
)

# Print the test statistic and p-value.
print("Mens Email vs No Email:")
print(mens_spend_test)

print("\nWomens Email vs No Email:")
print(womens_spend_test)

Mens Email vs No Email:
TtestResult(statistic=np.float64(5.300140358411669), pvalue=np.float64(1.1638149682254823e-07), df=np.float64(36671.15797736051))

Womens Email vs No Email:
TtestResult(statistic=np.float64(3.256371576025392), pvalue=np.float64(0.0011293971023632501), df=np.float64(40064.88440653067))


In [11]:
import numpy as np
from scipy import stats

# This function calculates:
# 1. the average spend difference between treatment and control
# 2. the uncertainty around that difference
# 3. a 95% confidence interval for the true treatment effect
def spend_lift_ci(treated_spend, control_spend, confidence=0.95):

    # Average spend in each group
    treated_mean = treated_spend.mean()
    control_mean = control_spend.mean()

    # Our observed treatment effect:
    # average treatment spend - average control spend
    lift = treated_mean - control_mean

    # Standard error tells us how much uncertainty there is
    # in the difference between the two sample averages.
    se = np.sqrt(
        treated_spend.var(ddof=1) / len(treated_spend)
        +
        control_spend.var(ddof=1) / len(control_spend)
    )

    # For a 95% confidence interval, this gives us
    # the critical value from the normal distribution.
    z = stats.norm.ppf(1 - (1 - confidence) / 2)

    # Build the lower and upper limits.
    lower = lift - z * se
    upper = lift + z * se

    return lift, lower, upper


# Calculate the confidence interval for Men's Email.
mens_ci = spend_lift_ci(
    mens["spend"],
    control["spend"]
)

# Calculate the confidence interval for Women's Email.
womens_ci = spend_lift_ci(
    womens["spend"],
    control["spend"]
)

print("Mens Email spend lift and 95% CI:")
print(mens_ci)

print("\nWomens Email spend lift and 95% CI:")
print(womens_ci)

Mens Email spend lift and 95% CI:
(np.float64(0.7698271558945368), np.float64(0.48514912892878365), np.float64(1.0545051828602898))

Womens Email spend lift and 95% CI:
(np.float64(0.4244122159365967), np.float64(0.16896450721722805), np.float64(0.6798599246559653))


In [12]:
# Calculate how many customers received each treatment.
n_mens = len(mens)
n_womens = len(womens)

# Convert the average spend lift per customer
# into estimated total incremental revenue
# for the customers who actually received that treatment.

mens_incremental_revenue = (
    (mens["spend"].mean() - control["spend"].mean()) * n_mens
)

womens_incremental_revenue = (
    (womens["spend"].mean() - control["spend"].mean()) * n_womens
)

print("Estimated incremental revenue from Men's Email:")
print(mens_incremental_revenue)

print("\nEstimated incremental revenue from Women's Email:")
print(womens_incremental_revenue)

Estimated incremental revenue from Men's Email:
16402.707210644894

Estimated incremental revenue from Women's Email:
9076.904062235993


In [17]:
import pandas as pd

# Reload the original dataset into the variable `df`.
# This restores `df` back to being a pandas DataFrame
# instead of the single number it was accidentally changed to.

df = pd.read_csv("../data/raw/hillstrom.csv")

# Check that `df` is a DataFrame again.
print(type(df))

# Check that we still have all 64,000 rows and 12 columns.
print(df.shape)

<class 'pandas.DataFrame'>
(64000, 12)


In [18]:
# Group customers by their previous purchase channel.
# This lets us compare treatment effects for:
# - Phone customers
# - Web customers
# - Multichannel customers

channel_summary = (
    df.groupby(["channel", "segment"])
      .agg(
          customers=("spend", "size"),
          avg_spend=("spend", "mean"),
          conversion_rate=("conversion", "mean"),
          visit_rate=("visit", "mean")
      )
      .reset_index()
)

channel_summary

,channel,segment,customers,avg_spend,conversion_rate,visit_rate
0,Multichannel,Mens E-Mail,2577,1.824870,0.017074,0.211486
1,Multichannel,No E-Mail,2606,0.615741,0.006907,0.128550
2,Multichannel,Womens E-Mail,2579,1.773276,0.013959,0.175649
3,Phone,Mens E-Mail,9240,1.208527,0.010823,0.162771
4,Phone,No E-Mail,9327,0.644453,0.005361,0.087166
5,Phone,Womens E-Mail,9454,0.876926,0.007087,0.131796
6,Web,Mens E-Mail,9490,1.521835,0.012961,0.194415
7,Web,No E-Mail,9373,0.671386,0.005761,0.118852
8,Web,Womens E-Mail,9354,1.087703,0.009194,0.164529


In [19]:
# Create a small table where each row is one customer channel
# and each column is the average spend under one treatment.
#
# This makes it easy to compare:
# - No Email
# - Men's Email
# - Women's Email
# within each channel.

channel_spend = (
    df.groupby(["channel", "segment"])["spend"]
      .mean()
      .unstack()
)

# Calculate incremental spend relative to the No Email group.
#
# Example:
# Men's lift = average spend with Men's Email
#              minus average spend with No Email

channel_spend["Mens Lift"] = (
    channel_spend["Mens E-Mail"] - channel_spend["No E-Mail"]
)

channel_spend["Womens Lift"] = (
    channel_spend["Womens E-Mail"] - channel_spend["No E-Mail"]
)

channel_spend

segment,Mens E-Mail,No E-Mail,Womens E-Mail,Mens Lift,Womens Lift
channel,,,,,
Multichannel,1.824870,0.615741,1.773276,1.209129,1.157536
Phone,1.208527,0.644453,0.876926,0.564074,0.232474
Web,1.521835,0.671386,1.087703,0.850449,0.416317


In [20]:
# Group customers by their historical spending segment
# and by the treatment they received.
#
# We calculate average post-campaign spend for each combination.

history_spend = (
    df.groupby(["history_segment", "segment"])["spend"]
      .mean()
      .unstack()
)

# Calculate how much extra spend each email generated
# compared with sending no email.

history_spend["Mens Lift"] = (
    history_spend["Mens E-Mail"] - history_spend["No E-Mail"]
)

history_spend["Womens Lift"] = (
    history_spend["Womens E-Mail"] - history_spend["No E-Mail"]
)

history_spend

segment,Mens E-Mail,No E-Mail,Womens E-Mail,Mens Lift,Womens Lift
history_segment,,,,,
1) $0 - $100,1.055176,0.516893,1.142713,0.538283,0.625820
2) $100 - $200,1.127143,0.422895,0.881288,0.704249,0.458393
3) $200 - $350,1.469462,0.932881,0.609740,0.536581,-0.323141
4) $350 - $500,2.579208,1.010268,0.909378,1.568940,-0.100890
5) $500 - $750,1.641866,0.538856,1.824507,1.103010,1.285651
"6) $750 - $1,000",1.883680,0.349534,0.997791,1.534146,0.648257
"7) $1,000 +",3.491875,2.169808,4.676589,1.322067,2.506781


In [21]:
# For each historical spending segment, look only at the
# average spend under the three possible actions:
# - Men's Email
# - Women's Email
# - No Email

best_treatment_by_history = history_spend[
    ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]
].copy()

# idxmax(axis=1) finds the column with the largest value
# in each row.
#
# In simple words:
# "Which action produced the highest average spend
# for this customer group?"

best_treatment_by_history["Best Action"] = (
    best_treatment_by_history.idxmax(axis=1)
)

# Also calculate the highest observed average spend
# so we can see how strong that best action was.

best_treatment_by_history["Best Avg Spend"] = (
    best_treatment_by_history[
        ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]
    ].max(axis=1)
)

best_treatment_by_history

segment,Mens E-Mail,Womens E-Mail,No E-Mail,Best Action,Best Avg Spend
history_segment,,,,,
1) $0 - $100,1.055176,1.142713,0.516893,Womens E-Mail,1.142713
2) $100 - $200,1.127143,0.881288,0.422895,Mens E-Mail,1.127143
3) $200 - $350,1.469462,0.609740,0.932881,Mens E-Mail,1.469462
4) $350 - $500,2.579208,0.909378,1.010268,Mens E-Mail,2.579208
5) $500 - $750,1.641866,1.824507,0.538856,Womens E-Mail,1.824507
"6) $750 - $1,000",1.883680,0.997791,0.349534,Mens E-Mail,1.883680
"7) $1,000 +",3.491875,4.676589,2.169808,Womens E-Mail,4.676589


In [22]:
# Create a simple customer preference label based on
# whether the customer previously bought men's and/or women's products.
#
# This turns the two binary columns:
# mens
# womens
#
# into an easier-to-read business category.

def purchase_preference(row):
    if row["mens"] == 1 and row["womens"] == 0:
        return "Mens Only"
    elif row["mens"] == 0 and row["womens"] == 1:
        return "Womens Only"
    elif row["mens"] == 1 and row["womens"] == 1:
        return "Both"
    else:
        return "Neither"


# Create the new readable customer segment.
df["purchase_preference"] = df.apply(purchase_preference, axis=1)


# Calculate average spend under each treatment
# for each historical purchase-preference group.

preference_spend = (
    df.groupby(["purchase_preference", "segment"])["spend"]
      .mean()
      .unstack()
)


# Calculate treatment lift relative to No Email.

preference_spend["Mens Lift"] = (
    preference_spend["Mens E-Mail"] - preference_spend["No E-Mail"]
)

preference_spend["Womens Lift"] = (
    preference_spend["Womens E-Mail"] - preference_spend["No E-Mail"]
)

preference_spend

segment,Mens E-Mail,No E-Mail,Womens E-Mail,Mens Lift,Womens Lift
purchase_preference,,,,,
Both,3.005131,1.180144,1.342025,1.824986,0.161881
Mens Only,1.378162,0.695426,0.973781,0.682735,0.278354
Womens Only,1.106295,0.490564,1.122212,0.615731,0.631648


In [23]:
# Count how many customers belong to each
# historical purchase-preference group.

# This helps us make sure we are not drawing conclusions
# from a tiny group with only a few customers.

df["purchase_preference"].value_counts()

purchase_preference
Mens Only      28818
Womens Only    28734
Both            6448
Name: count, dtype: int64

In [24]:
# Cross-tabulate purchase preference against treatment group.
#
# This tells us how many customers from each preference group
# were randomly assigned to each email treatment.

pd.crosstab(
    df["purchase_preference"],
    df["segment"]
)

segment,Mens E-Mail,No E-Mail,Womens E-Mail
purchase_preference,,,
Both,2181,2149,2118
Mens Only,9558,9638,9622
Womens Only,9568,9519,9647


In [25]:
# Reuse our earlier helper function that calculates:
# - observed spend lift
# - lower bound of the 95% confidence interval
# - upper bound of the 95% confidence interval

def spend_lift_ci(treated_spend, control_spend, confidence=0.95):
    treated_mean = treated_spend.mean()
    control_mean = control_spend.mean()

    # Estimated treatment effect
    lift = treated_mean - control_mean

    # Standard error of the difference between the two means
    se = np.sqrt(
        treated_spend.var(ddof=1) / len(treated_spend)
        +
        control_spend.var(ddof=1) / len(control_spend)
    )

    # Critical value for a 95% confidence interval
    z = stats.norm.ppf(1 - (1 - confidence) / 2)

    lower = lift - z * se
    upper = lift + z * se

    return lift, lower, upper


# Go through each purchase-preference group separately.
for preference in ["Mens Only", "Womens Only", "Both"]:

    # Keep only customers in this preference group.
    group = df[df["purchase_preference"] == preference]

    # Separate control, men's email, and women's email customers.
    group_control = group[group["segment"] == "No E-Mail"]
    group_mens = group[group["segment"] == "Mens E-Mail"]
    group_womens = group[group["segment"] == "Womens E-Mail"]

    # Calculate confidence intervals for both treatments vs control.
    mens_result = spend_lift_ci(
        group_mens["spend"],
        group_control["spend"]
    )

    womens_result = spend_lift_ci(
        group_womens["spend"],
        group_control["spend"]
    )

    print(f"\nPreference group: {preference}")
    print("Mens Email lift and 95% CI:", mens_result)
    print("Womens Email lift and 95% CI:", womens_result)


Preference group: Mens Only
Mens Email lift and 95% CI: (np.float64(0.682735312299813), np.float64(0.24717206544842596), np.float64(1.1182985591512))
Womens Email lift and 95% CI: (np.float64(0.27835448170778665), np.float64(-0.12356183001227383), np.float64(0.6802707934278471))

Preference group: Womens Only
Mens Email lift and 95% CI: (np.float64(0.6157308065834535), np.float64(0.2696778413087731), np.float64(0.9617837718581339))
Womens Email lift and 95% CI: (np.float64(0.6316479517709466), np.float64(0.29513352100515206), np.float64(0.9681623825367411))

Preference group: Both
Mens Email lift and 95% CI: (np.float64(1.824986420861755), np.float64(0.49016108990282037), np.float64(3.1598117518206896))
Womens Email lift and 95% CI: (np.float64(0.16188124260971226), np.float64(-0.8133372445223764), np.float64(1.1370997297418008))
